## 类内部常见三种方法：
| 方法类型 | 第一个参数  | 调用方式             | 常见用途    |
| ---- | ------ | ---------------- | ------- |
| 实例方法 | `self` | `obj.method()`   | 访问对象数据  |
| 类方法  | `cls`  | `Class.method()` | 访问类级别数据 |
| 静态方法 | 无固定参数  | `Class.method()` | 工具函数    |


## 装饰器装饰类内部的方法
- 实例方法会自动传入 self
- 类方法会自动传入 cls
- 静态方法不会自动传入特殊参数
- 普通参数也要继续传递



In [ ]:
from functools import wraps


def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"开始执行：{func.__name__}")

        result = func(*args, **kwargs)

        print(f"执行结束：{func.__name__}")

        return result

    return wrapper


# 装饰实例方法
class UserService:
    def __init__(self, username: str):
        self.username = username

    @log_call
    def create_user(self, age: int):
        print(f"创建用户：{self.username}，年龄：{age}")
        return {"username": self.username, "age": age}

service = UserService("张三")
result = service.create_user(18)
print(result)

"""
简单分析：

"""
class UserServiceB:
    @log_call
    def create_user(self, data):
        ...

    @log_call
    def update_user(self, user_id, data):
        ...

    @log_call
    def delete_user(self, user_id):
        ...

""" 
# 适合做
- Service 层日志
- Repository 层 SQL 调用统计
- 业务方法耗时统计
- 异常捕获
- 审计日志
- 权限检查
"""


开始执行：create_user
创建用户：张三，年龄：18
执行结束：create_user
{'username': '张三', 'age': 18}


In [1]:
# 装饰类方法
from functools import wraps


def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"开始执行：{func.__name__}")

        result = func(*args, **kwargs)

        print(f"执行结束：{func.__name__}")

        return result

    return wrapper


class Config:
    app_name = "订单系统"

    """
    为什么是这个顺序？
    @classmethod
    @log_call
    def get_app_name(cls):
        ...

    1.先用 log_call 装饰普通函数
    2.再把它变成类方法
    这样 cls 才能正常自动传入。
    """
    @classmethod
    @log_call
    def get_app_name(cls):
        print(f"当前应用：{cls.app_name}")
        return cls.app_name


result = Config.get_app_name()

print(result)


"""  
适合：
类级别配置读取
枚举校验
工厂方法
从配置创建对象
读取类共享资源

"""


开始执行：get_app_name
当前应用：订单系统
执行结束：get_app_name
订单系统


'  \n适合：\n类级别配置读取\n枚举校验\n工厂方法\n从配置创建对象\n读取类共享资源\n\n'

In [ ]:
# 装饰静态方法
# 静态方法也要注意装饰顺序。
# 推荐写法

from functools import wraps


def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"开始执行：{func.__name__}")

        result = func(*args, **kwargs)

        print(f"执行结束：{func.__name__}")

        return result

    return wrapper


class StringUtils:

    @staticmethod
    @log_call
    def normalize_phone(phone: str) -> str:
        return phone.replace(" ", "").replace("-", "")


result = StringUtils.normalize_phone("138-0000-8888")

print(result)


""" 
为什么是这个顺序？
@staticmethod
@log_call
def normalize_phone(phone: str):
    ...

等价于：
normalize_phone = staticmethod(log_call(normalize_phone))
先装饰函数，再变成静态方法。

静态方法适合放一些无状态工具逻辑：
适合：
- 字符串清洗
- 日期格式转换
- 金额格式化
- 参数预处理
- 简单计算工具
"""
class DateUtils:

    @staticmethod
    @log_call
    def format_date(date_str: str) -> str:
        return date_str.replace("/", "-")


In [ ]:
# 三种方法放在一起的完整案例
from functools import wraps


def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"\n开始执行：{func.__name__}")

        result = func(*args, **kwargs)

        print(f"执行结束：{func.__name__}")

        return result

    return wrapper


class UserService:
    system_name = "用户中心"

    def __init__(self, username: str):
        self.username = username

    @log_call
    def create_user(self, age: int):
        print(f"实例方法：创建用户 {self.username}，年龄 {age}")
        return {"username": self.username, "age": age}

    @classmethod
    @log_call
    def get_system_name(cls):
        print(f"类方法：当前系统是 {cls.system_name}")
        return cls.system_name

    @staticmethod
    @log_call
    def normalize_username(username: str):
        print("静态方法：清洗用户名")
        return username.strip().lower()


service = UserService("ZhangSan")

print(service.create_user(18))

print(UserService.get_system_name())

print(UserService.normalize_username("  ADMIN  "))


In [2]:
# 企业级装饰器案例：统计耗时
import time
from functools import wraps


def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()

        try:
            return func(*args, **kwargs)
        finally:
            end = time.perf_counter()
            cost = end - start
            print(f"{func.__name__} 执行耗时：{cost:.6f} 秒")

    return wrapper

# 使用
class OrderService:

    @timer
    def create_order(self, user_id: int):
        print(f"创建订单，用户ID：{user_id}")

    @classmethod
    @timer
    def get_service_name(cls):
        return "订单服务"

    @staticmethod
    @timer
    def calculate_total(price: float, count: int):
        return price * count


service = OrderService()

service.create_order(1001)
print(OrderService.get_service_name())
print(OrderService.calculate_total(99.9, 3))


创建订单，用户ID：1001
create_order 执行耗时：0.000847 秒
get_service_name 执行耗时：0.000593 秒
订单服务
calculate_total 执行耗时：0.000896 秒
299.70000000000005


In [ ]:
# 企业级装饰器案例：权限控制
from functools import wraps


def require_role(required_role: str):
    def decorator(func):
        @wraps(func)
        def wrapper(self, user_role: str, *args, **kwargs):
            if user_role != required_role:
                raise PermissionError(f"权限不足，需要角色：{required_role}")

            return func(self, user_role, *args, **kwargs)

        return wrapper

    return decorator

class AdminService:

    @require_role("admin")
    def delete_user(self, user_role: str, user_id: int):
        print(f"删除用户：{user_id}")


service = AdminService()

service.delete_user("admin", 1001)


In [ ]:
# 企业项目中推荐的装饰器顺序

class UserService:

    @timer
    @log_call
    def create_user(self):
        ...

# 类方法
# 推荐：
class Config:

    @classmethod
    @timer
    @log_call
    def load(cls):
        ...
# 静态方法
class StringUtils:

    @staticmethod
    @timer
    @log_call
    def clean(text: str):
        ...


<!-- 企业实践建议 -->
# 建议一：业务逻辑不要写进装饰器
## 装饰器适合做横切逻辑：
日志、耗时、权限、缓存、重试、事务、审计
不适合写核心业务：
创建订单、扣库存、支付、发货

# 建议二：实例方法最常用装饰器
企业 Service 层常见：
class OrderService:

    @transaction
    @audit_log
    @timer
    def create_order(self, data):
        ...

# 建议三：类方法适合工厂类、配置类
class DatabaseConfig:

    @classmethod
    @timer
    def from_env(cls):
        ...

建议四：静态方法适合纯工具函数
class MoneyUtils:

    @staticmethod
    @timer
    def yuan_to_cent(amount: float) -> int:
        return int(amount * 100)

最重要的规则
实例方法：装饰器直接放在 def 上面
类方法：@classmethod 通常放最外层
静态方法：@staticmethod 通常放最外层

In [ ]:
# 案例 1：配置类，从环境变量创建配置对象
# 适用场景
# 企业项目中，数据库连接、Redis 地址、API 密钥通常不会写死在代码里，而是来自环境变量。

import os
from dataclasses import dataclass


@dataclass
class DatabaseConfig:
    host: str
    port: int
    username: str
    password: str
    database: str

    @property
    def dsn(self) -> str:
        return (
            f"postgresql://{self.username}:{self.password}"
            f"@{self.host}:{self.port}/{self.database}"
        )

    @classmethod
    def from_env(cls):
        """
        从环境变量创建数据库配置对象
        """
        return cls(
            host=os.getenv("DB_HOST", "127.0.0.1"),
            port=int(os.getenv("DB_PORT", "5432")),
            username=os.getenv("DB_USER", "postgres"),
            password=os.getenv("DB_PASSWORD", "123456"),
            database=os.getenv("DB_NAME", "app_db"),
        )


# 模拟环境变量
os.environ["DB_HOST"] = "192.168.0.10"
os.environ["DB_PORT"] = "5432"
os.environ["DB_USER"] = "admin"
os.environ["DB_PASSWORD"] = "secret"
os.environ["DB_NAME"] = "order_db"


config = DatabaseConfig.from_env()

print(config)
print(config.host)
print(config.port)


In [ ]:
# 案例 2：工厂类，根据数据创建不同支付对象
# 适用场景
# 企业项目中经常会有多种业务类型：
""" 
微信支付
支付宝支付
银行卡支付
余额支付

如果到处写：

if pay_type == "wechat":
    ...
elif pay_type == "alipay":
    ...

代码会越来越乱。

这时可以用类方法做“工厂方法”。
"""


class WechatPay:
    def pay(self, amount: float):
        print(f"使用微信支付：{amount} 元")


class AliPay:
    def pay(self, amount: float):
        print(f"使用支付宝支付：{amount} 元")


class BankPay:
    def pay(self, amount: float):
        print(f"使用银行卡支付：{amount} 元")


class PaymentFactory:
    @classmethod
    def create(cls, pay_type: str):
        """
        根据支付类型创建支付对象
        """
        if pay_type == "wechat":
            return WechatPay()

        if pay_type == "alipay":
            return AliPay()

        if pay_type == "bank":
            return BankPay()

        raise ValueError(f"不支持的支付方式：{pay_type}")


pay_client = PaymentFactory.create("wechat")
pay_client.pay(99.9)

pay_client = PaymentFactory.create("alipay")
pay_client.pay(199.9)


In [ ]:
# 更企业级一点的写法：注册表模式
# 企业项目里更常见的是“注册表”。


class WechatPay:
    def pay(self, amount: float):
        print(f"使用微信支付：{amount} 元")


class AliPay:
    def pay(self, amount: float):
        print(f"使用支付宝支付：{amount} 元")


class BankPay:
    def pay(self, amount: float):
        print(f"使用银行卡支付：{amount} 元")


class PaymentFactory:
    payment_map = {
        "wechat": WechatPay,
        "alipay": AliPay,
        "bank": BankPay,
    }

    @classmethod
    def create(cls, pay_type: str):
        pay_class = cls.payment_map.get(pay_type)

        if pay_class is None:
            raise ValueError(f"不支持的支付方式：{pay_type}")

        return pay_class()


pay_client = PaymentFactory.create("bank")
pay_client.pay(300)
